In [ ]:
import pandas as pd
import requests
from ast import literal_eval
from datetime import datetime
import os
import json

from draw import draw

In [132]:
draw = pd.DataFrame(list(draw.items()), columns=['team', 'name'])
draw

,team,name
0,Spain,Matt
1,Colombia,Matt
2,Austria,Matt
3,Canada,Matt
4,Saudi Arabia,Matt
5,South Africa,Matt
6,France,Dan
7,Uruguay,Dan
8,Japan,Dan
9,Sweden,Dan


In [133]:
subtitles_file = 'assets/data/subtitles.csv'

if not os.path.exists(subtitles_file):
    subtitles = pd.DataFrame(columns=['date', 'subtitle'])
else:
    subtitles = pd.read_csv(subtitles_file)
subtitles

,date,subtitle
0,2026-05-30 16:07:26.050852,Spain and France sit atop the market as co-lea...
1,2026-05-30 16:10:21.854241,Spain and France sit atop the market as co-lea...
2,2026-05-30 16:10:28.053220,Spain and France sit atop the market as co-lea...
3,2026-05-30 16:10:32.281287,Spain and France sit atop the market as co-lea...
4,2026-05-30 16:10:42.948733,Spain and France sit atop the market as co-lea...
5,2026-05-31 16:19:27.109896,Spain and France sit atop the market as co-lea...


In [134]:
prices_file = 'assets/data/prices.csv'

if not os.path.exists(prices_file):
    existing_prices = pd.DataFrame(columns=['team', 'price', 'date'])
else:
    existing_prices = pd.read_csv(prices_file)
existing_prices

,team,price,date
0,Spain,0.1675,2026-05-30 16:07:26.050852
1,New Zealand,0.0005,2026-05-30 16:07:26.050852
2,Switzerland,0.0105,2026-05-30 16:07:26.050852
3,England,0.1115,2026-05-30 16:07:26.050852
4,France,0.1635,2026-05-30 16:07:26.050852
...,...,...,...
283,Saudi Arabia,0.0005,2026-05-31 16:19:27.109896
284,Austria,0.0055,2026-05-31 16:19:27.109896
285,Croatia,0.0085,2026-05-31 16:19:27.109896
286,Egypt,0.0025,2026-05-31 16:19:27.109896


In [135]:
slug = "world-cup-winner"

response = requests.get(
    "https://gamma-api.polymarket.com/events/slug/" + slug
)
event = response.json()

valid_at = event["updatedAt"]
valid_at = datetime.now() + pd.Timedelta(days=8)  # Use current date for better visualization

with open('assets/data/data.json', 'w') as outfile:
    json.dump({"subtitle": event["eventMetadata"]["context_description"]}, outfile)

markets = [
    {
        "team": team.get("groupItemTitle"),
        "price": float(literal_eval(team.get("outcomePrices", "[0,1]"))[0]),
        "date": valid_at,
    }
    for team in event["markets"] if team.get("groupItemTitle") in list(draw['team'])
]

prices = pd.DataFrame(markets)
extended_prices = pd.concat([existing_prices, prices], ignore_index=True)
extended_prices['date'] = pd.to_datetime(extended_prices['date'])
extended_prices.to_csv('assets/data/prices.csv', index=False)

subtitles = pd.concat([subtitles, pd.DataFrame([{"date": valid_at, "subtitle": event["eventMetadata"]["context_description"]}])], ignore_index=True)
subtitles.to_csv('assets/data/subtitles.csv', index=False)


In [136]:
extended_prices

,team,price,date
0,Spain,0.1675,2026-05-30 16:07:26.050852
1,New Zealand,0.0005,2026-05-30 16:07:26.050852
2,Switzerland,0.0105,2026-05-30 16:07:26.050852
3,England,0.1115,2026-05-30 16:07:26.050852
4,France,0.1635,2026-05-30 16:07:26.050852
...,...,...,...
331,Saudi Arabia,0.0005,2026-06-07 16:20:46.142533
332,Austria,0.0055,2026-06-07 16:20:46.142533
333,Croatia,0.0085,2026-06-07 16:20:46.142533
334,Egypt,0.0025,2026-06-07 16:20:46.142533


In [137]:
# Normalise prices by date
extended_prices['price'] = extended_prices.groupby('date')['price'].transform(lambda x: x / x.sum())   
extended_prices

,team,price,date
0,Spain,0.163574,2026-05-30 16:07:26.050852
1,New Zealand,0.000488,2026-05-30 16:07:26.050852
2,Switzerland,0.010254,2026-05-30 16:07:26.050852
3,England,0.108887,2026-05-30 16:07:26.050852
4,France,0.159668,2026-05-30 16:07:26.050852
...,...,...,...
331,Saudi Arabia,0.000488,2026-06-07 16:20:46.142533
332,Austria,0.005366,2026-06-07 16:20:46.142533
333,Croatia,0.008293,2026-06-07 16:20:46.142533
334,Egypt,0.002439,2026-06-07 16:20:46.142533


In [138]:
# draw = draw.merge(extended_prices, on='team')
combined = pd.merge(extended_prices, draw, on='team')
combined

,team,price,date,name
0,Spain,0.163574,2026-05-30 16:07:26.050852,Matt
1,New Zealand,0.000488,2026-05-30 16:07:26.050852,Daf
2,Switzerland,0.010254,2026-05-30 16:07:26.050852,Llyr
3,England,0.108887,2026-05-30 16:07:26.050852,Rhys
4,France,0.159668,2026-05-30 16:07:26.050852,Dan
...,...,...,...,...
331,Saudi Arabia,0.000488,2026-06-07 16:20:46.142533,Matt
332,Austria,0.005366,2026-06-07 16:20:46.142533,Matt
333,Croatia,0.008293,2026-06-07 16:20:46.142533,Jac
334,Egypt,0.002439,2026-06-07 16:20:46.142533,Daf


In [139]:
values = combined.groupby(['name', 'date'])['price'].sum().reset_index(name='price')
values['champion'] = values['price'] * 120
values['runner up'] = values['price'] * 20
values['total'] = values['champion'] + values['runner up']
values['date'] = pd.to_datetime(values['date'])
values = values.sort_values(['date', 'total'])
values

,name,date,price,champion,runner up,total
42,Neb,2026-05-30 16:07:26.050852,0.002930,0.351562,0.058594,0.410156
14,Jac,2026-05-30 16:07:26.050852,0.074219,8.906250,1.484375,10.390625
0,Daf,2026-05-30 16:07:26.050852,0.106445,12.773438,2.128906,14.902344
28,Llyr,2026-05-30 16:07:26.050852,0.115234,13.828125,2.304688,16.132812
49,Rhys,2026-05-30 16:07:26.050852,0.147461,17.695312,2.949219,20.644531
21,Joey,2026-05-30 16:07:26.050852,0.166016,19.921875,3.320312,23.242188
35,Matt,2026-05-30 16:07:26.050852,0.190430,22.851562,3.808594,26.660156
7,Dan,2026-05-30 16:07:26.050852,0.197266,23.671875,3.945312,27.617188
43,Neb,2026-05-30 16:10:21.854241,0.002927,0.351220,0.058537,0.409756
15,Jac,2026-05-30 16:10:21.854241,0.074146,8.897561,1.482927,10.380488


In [140]:
dates = sorted(values['date'].dt.strftime('%Y-%m-%dT%H:%M:%S').unique().tolist())

datasets = [
    {
        "name": name,
        "total": group.sort_values('date')['total'].round(2).tolist(),
        "champion": group.sort_values('date')['champion'].round(2).tolist(),
    }
    for name, group in values.groupby('name')
]

with open('assets/data/chart_data.json', 'w') as f:
    json.dump({"labels": dates, "datasets": datasets}, f)
